# Coleta e Atualizacao de Dados BCB

Notebook para coleta automatica de indicadores economicos do Banco Central.

**Funcionalidades:**
- Download inicial automatico (historico completo)
- Atualizacao incremental (apenas dados novos)
- Consolidacao em arquivos Parquet

**Fontes de Dados:**
- **SGS**: Series temporais (IBC-Br, IGP-M, Selic, Dolar/Euro PTAX, CDI)
- **Expectations**: Expectativas de mercado do Relatorio Focus (IPCA, PIB, Cambio, Selic)

## 1. Setup

In [ ]:
from pathlib import Path
from bacen.sgs import SGSCollector, INDICATORS, list_indicators

# Inicializar coletor
data_path = Path.cwd().parent / 'data'
collector = SGSCollector(data_path)

print(f"Coletor inicializado: {data_path}")
print(f"Indicadores SGS: {list_indicators()}")

## 2. Coleta de Dados

Detecta automaticamente se deve fazer download completo ou atualizacao incremental.

In [ ]:
# Coletar todos os indicadores
results = collector.collect_all()

## 3. Consolidacao

In [ ]:
# Consolidar e salvar dados
df_monthly, df_daily = collector.consolidate_all()

## 4. Status dos Indicadores

In [ ]:
# Exibir status de cada indicador
collector.get_status()

## 5. Coleta de Expectativas (Focus)

Coleta expectativas de mercado do Relatorio Focus do BCB.

In [ ]:
from bacen.expectations import ExpectationsCollector, EXPECTATIONS_CONFIG, list_indicators as list_exp_indicators

# Inicializar coletor de expectativas
exp_collector = ExpectationsCollector(data_path)

print(f"Indicadores de expectativas: {list_exp_indicators()}")

In [ ]:
# Coletar todas as expectativas
exp_results = exp_collector.collect_all()

In [ ]:
# Status das expectativas coletadas
exp_collector.get_status()

## 6. Visualizacao Rapida

In [ ]:
import matplotlib.pyplot as plt

if not df_monthly.empty and not df_daily.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Mensais - ultimos 24 meses
    df_monthly.tail(24).plot(ax=axes[0], marker='o', linewidth=2)
    axes[0].set_title('Indicadores Mensais - Ultimos 24 Meses', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Valor')
    axes[0].legend(fontsize=9, loc='best')
    axes[0].grid(True, alpha=0.3)

    # Diarios - ultimos 90 dias
    df_daily.tail(90).plot(ax=axes[1], linewidth=1.5)
    axes[1].set_title('Indicadores Diarios - Ultimos 90 Dias', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Valor')
    axes[1].set_xlabel('Data')
    axes[1].legend(fontsize=9, loc='best')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Dados insuficientes para visualizacao.")